# **PySpark MLlib – Classification**

### Bank Dataset

**Target column:** `deposit`

Goal: predict the target class using customer/bank-related information.

| Column      | Description                                                                 |
| ----------- | --------------------------------------------------------------------------- |
| `age`       | Age of the customer                                                         |
| `job`       | Type of job/occupation                                                      |
| `marital`   | Marital status                                                              |
| `education` | Education level                                                             |
| `default`   | Whether the customer has credit in default                                  |
| `balance`   | Average yearly balance in the bank account                                  |
| `housing`   | Whether the customer has a housing loan                                     |
| `loan`      | Whether the customer has a personal loan                                    |
| `contact`   | Communication method used to contact the customer                           |
| `day`       | Day of the month when the customer was contacted                            |
| `month`     | Month when the customer was contacted                                       |
| `duration`  | Duration of the last contact                                                |
| `campaign`  | Number of contacts performed during the current campaign                    |
| `pdays`     | Number of days since the customer was last contacted in a previous campaign |
| `previous`  | Number of contacts performed before the current campaign                    |
| `poutcome`  | Outcome of the previous marketing campaign                                  |
| `deposit`   | **Target variable — whether the customer subscribed to a term deposit**     |


Predict whether a customer will subscribe to a bank term deposit based on their personal information, financial information, and campaign interaction.

Imagine a bank wants to contact customers and convince them to invest in a term deposit. Instead of contacting every customer randomly, we can use historical customer and campaign data to build a Machine Learning model that predicts which customers are more likely to subscribe.

## 1. Import Libraries and Create SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = SparkSession.builder \
    .appName("Bank Classification") \
    .master("local[*]") \
    .getOrCreate()

## 2. Load the Dataset

In [2]:
df = spark.read.csv("bank.csv",header=True,inferSchema=True)

In [3]:
df.show(5)

+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|age|       job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
| 59|    admin.|married|secondary|     no|   2343|    yes|  no|unknown|  5|  may|    1042|       1|   -1|       0| unknown|    yes|
| 56|    admin.|married|secondary|     no|     45|     no|  no|unknown|  5|  may|    1467|       1|   -1|       0| unknown|    yes|
| 41|technician|married|secondary|     no|   1270|    yes|  no|unknown|  5|  may|    1389|       1|   -1|       0| unknown|    yes|
| 55|  services|married|secondary|     no|   2476|    yes|  no|unknown|  5|  may|     579|       1|   -1|       0| unknown|    yes|
| 54|    admin.|married| tertiary|     no|    184|     no|  no|unknown|  5| 

In [ ]:
df.printSchema()

## 3. Identify Features and Target

Target/Label: `deposit`

The target is what we want the model to predict.

In [ ]:
target_col = "deposit"

feature_cols = [c for c in df.columns if c != target_col]

In [ ]:
print("Target:", target_col)
print("Features:", feature_cols)

## 4. Convert Categorical Columns

Machine Learning algorithms need numerical input.

`StringIndexer` converts categorical values into numeric indexes.

In [ ]:
categorical_cols = [field.name for field in df.schema.fields
    if field.name != target_col and field.dataType.simpleString() == "string"]

numeric_cols = [
    field.name for field in df.schema.fields
    if field.name != target_col and field.dataType.simpleString() != "string"]

In [ ]:
print("Categorical columns:", categorical_cols)

In [ ]:
print("Numeric columns:", numeric_cols)

## 5. Create Label and Feature Transformers

In [ ]:
label_indexer = StringIndexer(inputCol=target_col,outputCol="label",handleInvalid="keep")

indexers = [StringIndexer(inputCol=c, outputCol=c + "_index", handleInvalid="keep")
    for c in categorical_cols]

indexed_features = [c + "_index" for c in categorical_cols] + numeric_cols

In [ ]:
assembler = VectorAssembler(inputCols=indexed_features,outputCol="features",handleInvalid="keep")

## 6. Create Logistic Regression Model

Logistic Regression is a common algorithm for **classification**.

Here the model predicts one class of the target versus another.

In [ ]:
lr = LogisticRegression(featuresCol="features",labelCol="label")

## 7. Create Pipeline and Split the Data

In [ ]:
pipeline = Pipeline(stages=indexers + [label_indexer, assembler, lr])

train_data, test_data = df.randomSplit([0.8, 0.2],seed=42)

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())

## 8. Train the Model

In [ ]:
model = pipeline.fit(train_data)

print("Model trained successfully!")

## 9. Make Predictions

In [ ]:
predictions = model.transform(test_data)

predictions.select(target_col, "label", "prediction", "probability").show(10, truncate=False)

## 10. Evaluate the Classification Model

In [ ]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1")

In [ ]:
accuracy = accuracy_evaluator.evaluate(predictions)
f1 = f1_evaluator.evaluate(predictions)

print("Accuracy:", accuracy)
print("F1 Score:", f1)

### Classification Workflow

CSV → DataFrame → Categorical Encoding → Feature Vector → Train/Test Split → Logistic Regression → Prediction → Evaluation

### Important Terms

- **Feature:** Input variable used for prediction.
- **Label:** Target class we want to predict.
- **StringIndexer:** Converts categories into numerical indexes.
- **VectorAssembler:** Combines input columns into one feature vector.
- **Logistic Regression:** Classification algorithm.
- **Accuracy:** Percentage of correct predictions.
- **F1 Score:** Combines precision and recall into one score.

### Simple idea

**Customer information → MLlib Classification Model → Predicted Class**

In [ ]:
spark.stop()